# Acsis — Notebook 1: Research Agent (Phase 1)

**Wakasa Labs · Nairobi, Kenya · 2026**

This notebook builds and tests the core RETRIEVE → VERIFY loop.
Acsis searches the web, cross-references sources, scores confidence,
and returns verified facts with citations.

**Runtime:** Kaggle CPU or Google Colab (no GPU needed for Phase 1)

**What we build:**
1. Install dependencies
2. Test web search (DuckDuckGo, no API key)
3. Test arXiv search
4. Test Wikipedia
5. Full research pipeline on real questions
6. Confidence scoring
7. Memory storage (ChromaDB)

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
!pip install aiohttp chromadb sentence-transformers nest-asyncio -q
print('Dependencies installed')

: 

In [ ]:
# ── 2. Setup async in notebook ─────────────────────────────────────────────
import nest_asyncio
nest_asyncio.apply()
import asyncio, logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
print('Async ready')

In [ ]:
# ── 3. Clone Acsis repo ────────────────────────────────────────────────────
import os, sys
if not os.path.exists('acsis_research'):
    !git clone https://github.com/LensenWakasa/acsis-research.git acsis_research
sys.path.insert(0, 'acsis_research')

# If running standalone (no repo yet), define inline versions:
# (The cells below define everything inline so it works before the repo exists)

In [ ]:
# ── 4. Standalone Research Tool (no repo needed) ────────────────────────────
import aiohttp
import re
from dataclasses import dataclass, field

@dataclass
class ResearchResult:
    query: str
    facts: list
    sources: list
    confidence: float
    contradictions: list

async def search_duckduckgo(query: str) -> dict:
    """Free web search — no API key needed."""
    async with aiohttp.ClientSession() as session:
        params = {'q': query, 'format': 'json', 'no_html': 1}
        async with session.get('https://api.duckduckgo.com/', params=params) as resp:
            data = await resp.json(content_type=None)
    facts = []
    sources = []
    if data.get('Abstract'):
        facts.append(data['Abstract'])
        sources.append(data.get('AbstractURL', ''))
    for topic in data.get('RelatedTopics', [])[:4]:
        if isinstance(topic, dict) and topic.get('Text'):
            facts.append(topic['Text'][:300])
            if topic.get('FirstURL'):
                sources.append(topic['FirstURL'])
    return {'facts': facts, 'sources': sources}

async def search_arxiv(query: str, max_results: int = 3) -> dict:
    """arXiv scientific paper search — free API."""
    async with aiohttp.ClientSession() as session:
        params = {'search_query': f'all:{query}', 'max_results': max_results}
        async with session.get('http://export.arxiv.org/api/query', params=params) as resp:
            text = await resp.text()
    abstracts = re.findall(r'<summary>(.*?)</summary>', text, re.DOTALL)
    ids = re.findall(r'<id>http.*?/abs/(.*?)</id>', text)
    facts = [f'[arXiv] {ab.strip()[:300]}' for ab in abstracts[:max_results]]
    sources = [f'https://arxiv.org/abs/{pid}' for pid in ids[:max_results]]
    return {'facts': facts, 'sources': sources}

async def search_wikipedia(query: str) -> dict:
    """Wikipedia summary — free API."""
    title = query.replace(' ', '_')[:50]
    async with aiohttp.ClientSession() as session:
        url = f'https://en.wikipedia.org/api/rest_v1/page/summary/{title}'
        async with session.get(url) as resp:
            if resp.status != 200:
                return {'facts': [], 'sources': []}
            data = await resp.json()
    extract = data.get('extract', '')
    facts = [s.strip() for s in extract.split('. ')[:4] if len(s) > 30]
    source = data.get('content_urls', {}).get('desktop', {}).get('page', '')
    return {'facts': facts, 'sources': [source]}

def score_confidence(facts: list, n_sources: int, contradictions: list) -> float:
    if not facts or not n_sources:
        return 0.1
    base = min(0.90, 0.3 + n_sources * 0.10)
    penalty = len(contradictions) * 0.15
    return round(max(0.1, base - penalty), 2)

async def investigate(question: str) -> ResearchResult:
    """Full research pipeline on a question."""
    print(f'\n🔍 Investigating: {question}')
    ddg, arxiv, wiki = await asyncio.gather(
        search_duckduckgo(question),
        search_arxiv(question),
        search_wikipedia(question),
    )
    all_facts = ddg['facts'] + arxiv['facts'] + wiki['facts']
    all_sources = [s for s in ddg['sources'] + arxiv['sources'] + wiki['sources'] if s]
    all_facts = list(dict.fromkeys(all_facts))[:15]
    all_sources = list(dict.fromkeys(all_sources))[:8]
    confidence = score_confidence(all_facts, len(all_sources), [])
    return ResearchResult(
        query=question,
        facts=all_facts,
        sources=all_sources,
        confidence=confidence,
        contradictions=[],
    )

print('Research tools defined ✓')

In [ ]:
# ── 5. TEST: Research a real question ─────────────────────────────────────
question = 'What causes malaria and how is it transmitted?'
result = await investigate(question)

print(f'\n{'='*60}')
print(f'QUESTION: {result.query}')
print(f'CONFIDENCE: {result.confidence:.0%}')
print(f'SOURCES ({len(result.sources)}):')
for s in result.sources:
    print(f'  • {s}')
print(f'\nFACTS ({len(result.facts)}):')
for i, f in enumerate(result.facts[:6]):
    print(f'  {i+1}. {f[:200]}')
print(f'{'='*60}')

In [ ]:
# ── 6. TEST: Scientific question (hits arXiv) ─────────────────────────────
sci_q = 'What is CRISPR gene editing and how does it work?'
sci_result = await investigate(sci_q)

print(f'Confidence: {sci_result.confidence:.0%}')
print(f'Sources: {len(sci_result.sources)}')
print(f'Facts: {len(sci_result.facts)}')
print()
for f in sci_result.facts[:4]:
    print(f'  → {f[:200]}')

In [ ]:
# ── 7. Memory: Store what we learned ──────────────────────────────────────
# ChromaDB: persistent semantic memory
try:
    import chromadb
    from sentence_transformers import SentenceTransformer
    client = chromadb.PersistentClient(path='./acsis_memory_nb1')
    collection = client.get_or_create_collection('knowledge')
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    print('ChromaDB + embedder loaded ✓')

    # Store all facts from both research results
    import uuid
    all_to_store = result.facts + sci_result.facts
    for fact in all_to_store:
        if len(fact) > 20:
            emb = embedder.encode(fact, normalize_embeddings=True).tolist()
            collection.add(documents=[fact], embeddings=[emb], ids=[str(uuid.uuid4())])

    print(f'Stored {collection.count()} facts in memory')

    # Test retrieval
    query = 'How does malaria spread?'
    q_emb = embedder.encode(query, normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=[q_emb], n_results=3)
    print(f'\nMemory search: "{query}"')
    for doc in results['documents'][0]:
        print(f'  → {doc[:150]}')

except ImportError:
    print('ChromaDB not installed. Run: pip install chromadb sentence-transformers')

In [ ]:
# ── 8. Batch research (multiple questions) ─────────────────────────────────
questions = [
    'What are the main causes of poverty in sub-Saharan Africa?',
    'How does the human immune system fight viral infections?',
    'What is quantum entanglement?',
]

batch_results = await asyncio.gather(*[investigate(q) for q in questions])

print('\nBATCH RESEARCH SUMMARY')
print('='*50)
for r in batch_results:
    print(f'Q: {r.query[:60]}')
    print(f'   Confidence: {r.confidence:.0%} | Sources: {len(r.sources)} | Facts: {len(r.facts)}')
    if r.facts:
        print(f'   Top fact: {r.facts[0][:120]}')
    print()

In [ ]:
# ── 9. PHASE 1 PASS CRITERION ─────────────────────────────────────────────
print('PHASE 1 VALIDATION')
print('='*40)

checks = [
    ('Web search working', len(result.facts) > 0),
    ('arXiv search working', any('[arXiv]' in f for f in sci_result.facts)),
    ('Confidence scoring working', 0 < result.confidence <= 1),
    ('Multiple sources retrieved', len(result.sources) >= 2),
    ('Batch research working', all(len(r.facts) > 0 for r in batch_results)),
]

passed = 0
for name, ok in checks:
    status = '✓ PASS' if ok else '✗ FAIL'
    print(f'  {status} | {name}')
    if ok: passed += 1

print(f'\nResult: {passed}/{len(checks)} checks passed')
print('PHASE 1 COMPLETE ✓' if passed == len(checks) else 'FIX FAILING CHECKS')